In [1]:
import os
import sys

os.environ.pop("SPARK_HOME", None)
os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)

os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.19.10-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ.get("PATH", "")

os.environ["PYSPARK_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))

Python: c:\Users\MSIS\AppData\Local\Programs\Python\Python312\python.exe
JAVA_HOME: C:\Program Files\Microsoft\jdk-17.0.19.10-hotspot
SPARK_HOME: None
PYSPARK_PYTHON: c:\Users\MSIS\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Fire Example") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark working")

Spark version: 3.5.1
Spark working


In [3]:
import pyspark
import py4j
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

filepath = r"D:\spark_abd\sf-fire-calls.csv"

In [4]:
def create_sparkSession():
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("Fire Example") \
        .getOrCreate()
    return spark


In [5]:
spark = create_sparkSession()

In [6]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    df1 = df.select('CallType', 'CallDate', 'City', 'Zipcode', 'Neighborhood', 'Delay')
    return df1


In [7]:
df = create_dataframe(spark, filepath)
df.printSchema()
df.show(5)

root
 |-- CallType: string (nullable = true)
 |-- CallDate: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)

+----------------+----------+----+-------+--------------------+---------+
|        CallType|  CallDate|City|Zipcode|        Neighborhood|    Delay|
+----------------+----------+----+-------+--------------------+---------+
|  Structure Fire|01/11/2002|  SF|  94109|     Pacific Heights|     2.95|
|Medical Incident|01/11/2002|  SF|  94124|Bayview Hunters P...|      4.7|
|Medical Incident|01/11/2002|  SF|  94102|          Tenderloin|2.4333334|
|    Vehicle Fire|01/11/2002|  SF|  94110|      Bernal Heights|      1.5|
|          Alarms|01/11/2002|  SF|  94109|    Western Addition|3.4833333|
+----------------+----------+----+-------+--------------------+---------+
only showing top 5 rows



In [8]:
def clean_dataset(df):
    df1 = df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2 = df1.withColumn('Year', year(col('Date'))).withColumn('Month', month(col('Date'))).withColumn('Week', weekofyear(col('Date')))
    return df2


In [9]:
spark = create_sparkSession()

In [10]:
fire_df = clean_dataset(df)
fire_df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [11]:
def mapSeason(data):
    if 3 <= data <= 5:
        return 'Spring'
    elif 6 <= data <= 8:
        return 'Summer'
    elif 9 <= data <= 11:
        return 'Fall'
    else:
        return 'Winter'

season_udf = udf(mapSeason, StringType())
clean_df = fire_df.withColumn('Season', season_udf(col('Month')))
clean_df.show()

Py4JJavaError: An error occurred while calling o78.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3.0 failed 1 times, most recent failure: Lost task 0.0 in stage 3.0 (TID 13) (DESKTOP-EO308VR executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:99)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(Int

In [12]:
season_expr = (
    when(col("Month").isNull(), "Unknown")
    .when(col("Month").between(3, 5), "Spring")
    .when(col("Month").between(6, 8), "Summer")
    .when(col("Month").between(9, 11), "Fall")
    .otherwise("Winter")
)

clean_df = fire_df.withColumn("Season", season_expr)
clean_df.select("Month", "Season").show()

+-----+------+
|Month|Season|
+-----+------+
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
|    1|Winter|
+-----+------+
only showing top 20 rows



In [13]:
fire_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [14]:
clean_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

1. Get yearly count of fire calls

In [15]:
clean_df.select('Year').groupBy('Year').count().orderBy('Year',ascending = True).show()

+----+-----+
|Year|count|
+----+-----+
|2000| 5459|
|2001| 7713|
|2002| 8090|
|2003| 8499|
|2004| 8283|
|2005| 8282|
|2006| 8174|
|2007| 8255|
|2008| 8869|
|2009| 8789|
|2010| 9341|
|2011| 9735|
|2012| 9674|
|2013|10020|
|2014|10775|
|2015|11458|
|2016|11609|
|2017|12135|
|2018|10136|
+----+-----+



2.Different types of fire calls in 2018

In [16]:
clean_df.select('CallType').where(col('Year')==2018).distinct().show(truncate=False)

+-------------------------------+
|CallType                       |
+-------------------------------+
|Elevator / Escalator Rescue    |
|Alarms                         |
|Odor (Strange / Unknown)       |
|Citizen Assist / Service Call  |
|HazMat                         |
|Vehicle Fire                   |
|Other                          |
|Outside Fire                   |
|Traffic Collision              |
|Assist Police                  |
|Gas Leak (Natural and LP Gases)|
|Water Rescue                   |
|Electrical Hazard              |
|Structure Fire                 |
|Medical Incident               |
|Fuel Spill                     |
|Smoke Investigation (Outside)  |
|Train / Rail Incident          |
|Explosion                      |
|Suspicious Package             |
+-------------------------------+



3. Which week in the year in 2018 had the most fire calls?


In [17]:
import pyspark.sql.functions
import pyspark.sql.types
clean_df.select('Week').where(col('Year')==2018).groupBy('week').count().orderBy('count',ascending = False).collect()[0][0]

22

4. Get monthly count of fire calls based on year

In [18]:
year = int(input("Enter year bro"))
clean_df.select('Month').where(col('Year')==year).groupBy('Month').count().show()
#clean_df.select('Year').groupBy('Month').count().orderBy('Year',ascending = True).show()

+-----+-----+
|Month|count|
+-----+-----+
|    1| 1009|
|    6|  959|
|    3| 1007|
|    5| 1012|
|    4|  905|
|    2|  917|
|    9|  994|
|    8|  921|
|    7|  930|
|   10|  912|
|   12|  962|
|   11|  930|
+-----+-----+



5. Give monthly report of fire call types for selected year

In [20]:
year = int(input("Again enter year bro"))
clean_df.select('Month','CallType').where(col('Year')==year).groupBy('Month','CallType').count().orderBy('Month',ascending=True).show()

+-----+--------------------+-----+
|Month|            CallType|count|
+-----+--------------------+-----+
|    1|               Other|   11|
|    1|        Vehicle Fire|    2|
|    1|Odor (Strange / U...|    5|
|    1|      Structure Fire|  138|
|    1|Industrial Accidents|    1|
|    1|Citizen Assist / ...|   12|
|    1|   Electrical Hazard|    1|
|    1|Elevator / Escala...|    1|
|    1|        Outside Fire|   13|
|    1|    Medical Incident|  450|
|    1|              Alarms|   88|
|    1|Smoke Investigati...|    4|
|    2|        Outside Fire|    2|
|    2|              Alarms|   61|
|    2|      Structure Fire|  129|
|    2|Smoke Investigati...|    2|
|    2|        Vehicle Fire|    1|
|    2|Gas Leak (Natural...|    1|
|    2|Odor (Strange / U...|    1|
|    2|    Medical Incident|  452|
+-----+--------------------+-----+
only showing top 20 rows



6. Give top five fire call types for every season of selected year (seasons are like Spring, summer, fall
winter etc).

In [21]:
grp_df = clean_df.select('CallType').groupBy('Season')
grp_df

GroupedData[grouping expressions: [Season], value: [CallType: string], type: GroupBy]

7. Whether fire type calls are seasonal?
